# Планирование последовательностей интенций поверх FB — воспроизведение

Итог уже известен и получен на CPU: **метод проигрывает бейзлайну** (0.730
против 0.797, парная разность −0.067 с CI [−0.080, −0.050]). Подробности в
`REPORT.md`.

Смысл этого ноутбука — не пересчитать то же самое быстрее, а закрыть
**единственный открытый вопрос**, который CPU не потянул.

Диагноз из отчёта: узкое место — качество попарных оценок достижимости.
Корреляция стоимости с истинным расстоянием растёт с размером набора узла:

| членов в наборе | 1 | 8 | 16 | 32 |
|---|---|---|---|---|
| корреляция | 0.30 | 0.455 | 0.505 | 0.528 |

Все числа отчёта получены при **8** членах и 300 узлах — больше на CPU не
помещалось. Вопрос: если дать рёбрам лучшее качество (32 члена, 1000 узлов),
сократится ли разрыв?

Честное ожидание: скорее нет. Даже 0.528 далеко от 0.75, которые даёт прямая
оценка до цели. Но это предсказание, а не замер.

Времени займёт около полутора часов: четыре прогона по 300 эпизодов каждый.
Граф с полной конфигурацией строится один раз и кэшируется, поэтому три
последних прогона его переиспользуют.


## 1. Установка

In [ ]:
import os

# Абсолютный путь и проверка на существование — иначе повторный запуск ячейки
# клонирует репозиторий ВНУТРЬ уже скачанного и уходит на уровень глубже. Так
# уже случалось: рабочей оказывалась вложенность из трёх одинаковых папок.
REPO = '/content/fb-multi-intention-planning'
URL = 'https://github.com/2FIVE192/fb-multi-intention-planning.git'

if os.path.isdir(os.path.join(REPO, '.git')):
    print('репозиторий уже склонирован, обновляю')
    !cd {REPO} && git pull --ff-only && git submodule update --init --recursive
else:
    !git clone --recursive {URL} {REPO}

%cd {REPO}
!pip install -q -r requirements-colab.txt
print('\nрабочая директория:', os.getcwd())

In [ ]:
import os
import subprocess
import sys

# JAX по умолчанию занимает ~75% видеопамяти при первом же использовании. Если
# это сделает ядро ноутбука, дочерним процессам памяти уже не достанется.
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')


def run(*args, env=None, quiet=False):
    """Запускает скрипт репозитория отдельным процессом.

    Вывод читается построчно и печатается заново через print. Это не
    придирчивость: `subprocess.run` наследует файловые дескрипторы ядра, а в
    Colab вывод ячейки — объект Python поверх ZMQ, а не настоящий дескриптор.
    Без перепечатки весь вывод дочернего процесса уходит в лог сервера.

    `-X faulthandler` заставляет Python напечатать стек при падении в нативном
    коде: без него SIGSEGV не оставляет вообще никаких следов. Именно так и был
    найден источник падения на Colab.

    Функция намеренно ничего не возвращает: значение последнего выражения
    ячейки Jupyter печатает сам, и захваченный вывод дублировался бы сырой
    строкой с видимыми \\n.
    """
    command = [sys.executable, '-X', 'faulthandler', '-u', *args]
    if not quiet:
        print('$', ' '.join(command[4:]), flush=True)

    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1, env=env,
    )
    for line in process.stdout:
        if not quiet:
            print(line, end='', flush=True)
    code = process.wait()

    if code == 0:
        return

    hints = {-9: 'SIGKILL, почти всегда нехватка оперативной памяти',
             -6: 'SIGABRT, падение в нативной библиотеке',
             -11: 'SIGSEGV, падение в нативной библиотеке'}
    raise RuntimeError(f'процесс завершился с кодом {code}'
                       + (f' ({hints[code]})' if code in hints else ''))


# --- Проверка окружения ----------------------------------------------------
# Про графику. OGBench безусловно создаёт mujoco.Renderer в MazeEnv.__init__ и
# сразу рендерит кадр, то есть требует рабочего OpenGL даже когда картинки не
# нужны. На Colab это давало SIGSEGV прямо в mujoco.MjrContext — и на egl, и на
# osmesa, причём только когда до создания среды успевал загрузиться jaxlib с
# CUDA. Подбор backend'а такую поломку не лечит: конфликтуют нативные библиотеки.
#
# Поэтому репозиторий подменяет рендерер заглушкой (fbplan/_upstream.py):
# кадры мы нигде не используем, а источник падения исчезает целиком. Если
# картинки всё же понадобятся, верните настоящий рендерер через
# FBPLAN_KEEP_RENDERER=1 — тогда backend снова придётся подбирать.

try:
    run('scripts/diagnose_env.py', '--probe', 'upstream_path')
except RuntimeError as exc:
    raise RuntimeError(
        f"""Среда не создаётся ({exc}).

Полная диагностика покажет, на каком шаге ломается:
    run('scripts/diagnose_env.py')

Она прогоняет пять проб, отличающихся одним шагом, каждую отдельным процессом
с faulthandler — первая упавшая и называет причину."""
    ) from exc

# GPU проверяем тоже в подпроцессе: ядро не должно трогать видеопамять вообще.
run('-c', 'import jax; print("устройства jax:", jax.devices()); '
          'assert jax.devices()[0].platform == "gpu", '
          '"GPU не подключён: Среда выполнения -> Сменить среду выполнения"')

print('\nокружение готово')

## 2. Данные

In [ ]:
!python scripts/download_datasets.py --datasets antmaze-medium-navigate-v0

## 3. Чекпоинты и настройки

In [ ]:
!pip -q install gdown
!python -m gdown --folder https://drive.google.com/drive/folders/1dKYhaDJH9lUREo-kUV3AwmTLrxvKO7Ek -O checkpoints

CHECKPOINT = 'checkpoints/medium'
ENV = 'ogbench-antmaze-medium-navigate-v0'

import os
assert os.path.isfile(os.path.join(CHECKPOINT, 'params.pkl')), 'чекпоинт не скачался'
print(sorted(os.listdir(CHECKPOINT)))

## 4. Проверки перед прогоном

Тесты логики планирования (чекпоинт не нужен, секунды) и калибровка масштабов
среды. Калибровка здесь заодно работает как первая настоящая проверка связки
«среда + датасет + чекпоинт»: если что-то не так, узнаем сейчас, а не через час.

In [ ]:
run('tests/test_planning.py')
run('scripts/calibrate.py')

## 5. Главный вопрос: помогает ли лучшее качество рёбер

Сравниваем конфигурацию из отчёта (300 узлов, 8 членов) с полной (1000 узлов,
32 члена). Всё остальное совпадает, включая отложенные сиды 1–3 — на них
подбора гиперпараметров не было.

Если разрыв с бейзлайном сократится — диагноз «дело в качестве рёбер» получает
количественное подтверждение и появляется понятное направление работы. Если
нет — значит упирается не в разрешение оценки, а в саму величину.

In [ ]:
COMMON = ['--checkpoint_dir', CHECKPOINT, '--env_name', ENV,
          '--methods', 'baseline,graph',
          '--seeds', '1,2,3', '--num_episodes', '20',
          '--replan_every', '20', '--execution', 'high',
          '--min_commit_steps', '40',
          '--tail_estimate', 'direct', '--plan_advantage_steps', '25',
          '--no_progress']

# А: ровно та конфигурация, которой получены числа отчёта (контроль).
run('scripts/run_eval.py', *COMMON,
    '--num_nodes', '300', '--num_members', '8', '--member_stride', '8',
    '--normalizer_references', '1000', '--tag', 'gpu_small')

# Б: полная конфигурация — рёбра максимального качества.
run('scripts/run_eval.py', *COMMON,
    '--num_nodes', '1000', '--num_members', '32', '--member_stride', '2',
    '--normalizer_references', '4000', '--tag', 'gpu_full')

## 6. Контрольная абляция: глубина плана

Отличие в одном флаге `--tail_estimate`: `dijkstra` — многошаговая композиция,
`direct` — план из одной подцели. На CPU было 0.47 против 0.69.

Обе ветки идут **без** `--plan_advantage_steps`, поэтому сравнивать их нужно
между собой, а не с прогоном «полная» из предыдущего пункта: там запас доверия
включён, и отличий от этих веток сразу два.

In [ ]:
for tail in ['dijkstra', 'direct']:
    run('scripts/run_eval.py', '--checkpoint_dir', CHECKPOINT, '--env_name', ENV,
        '--methods', 'graph', '--seeds', '1,2,3', '--num_episodes', '20',
        '--replan_every', '20', '--execution', 'high', '--min_commit_steps', '40',
        '--tail_estimate', tail,
        '--num_nodes', '1000', '--num_members', '32', '--member_stride', '2',
        '--normalizer_references', '4000', '--no_progress',
        '--tag', f'gpu_tail_{tail}')

## 7. Сводка

In [ ]:
import os
import sys

import pandas as pd

# Абсолютный путь, а не '.': если рантайм успел перезапуститься (например,
# кончилась квота GPU), рабочая папка возвращается в /content, и относительный
# путь ломает импорт. Ровно так сводка и упала.
REPO = '/content/fb-multi-intention-planning'
if os.path.isdir(REPO):
    os.chdir(REPO)
sys.path.insert(0, REPO)

from fbplan.stats import paired_comparison

TAGS = {
    'gpu_small': 'контроль: 300 узлов, 8 членов (конфигурация отчёта)',
    'gpu_full': 'полная: 1000 узлов, 32 члена',
    'gpu_tail_dijkstra': 'абляция: хвост по Дейкстре',
    'gpu_tail_direct': 'абляция: хвост одним запросом',
}

missing = []
for tag, title in TAGS.items():
    path = f'results/raw/{tag}_episodes.csv'
    if not os.path.exists(path):
        # О пропаже нужно сказать вслух: молчаливый пропуск выглядит как
        # «результатов нет», а не как «прогон не отработал».
        missing.append(tag)
        continue

    df = pd.read_csv(path)
    print(f'--- {title} ---')
    for method, success in df.groupby('method').success.mean().items():
        print(f'    {method:9s} {success:.3f}   ({len(df[df.method == method])} эпизодов)')

    if {'graph', 'baseline'} <= set(df.method.unique()):
        cmp = paired_comparison(df, 'graph', 'baseline')
        print(f'    парная разность {cmp["delta"]:+.3f} '
              f'[{cmp["ci_low"]:+.3f}, {cmp["ci_high"]:+.3f}] по {cmp["num_pairs"]} парам')
    print()

if missing:
    print(f'НЕТ РЕЗУЛЬТАТОВ для: {", ".join(missing)}')
    print('Соответствующие прогоны не отработали — ищите ошибку в ячейках выше.')
else:
    print('Все прогоны на месте.')